In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 96 samples and 7778 columns.
First few rows of the data:


,dead or last follow up (days),dead,ID,age at diagnosis,clinical stage,MYCN amplification,17q unbalance,patient number,probe ID tumor,RNA-seq data,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,2121.0,0.0,1,356,1,-,+,NaN,984760.0,+,...,8.361404,7.840448,8.372249,8.109869,6.421469,6.893357,9.263799,8.778961,7.866203,8.322789
1,1456.0,0.0,2,1197,1,-,-,NaN,1178899.0,+,...,9.172643,7.889168,9.005146,9.100367,7.104996,7.129575,9.759173,9.271679,9.520435,8.990730
2,590.0,0.0,4,298,1,-,-,NaN,2217756.0,+,...,7.959898,6.805186,8.079660,8.220222,6.292021,6.856885,9.079503,8.695052,7.957350,8.038179
3,1631.0,0.0,5,1852,1,-,-,NaN,2238721.0,+,...,7.889291,6.553900,8.233543,7.954751,6.139533,6.657749,9.003603,8.600086,7.712642,8.325577
4,54.0,0.0,6,95,1,-,-,NaN,2238819.0,+,...,9.119654,7.903819,9.356463,9.497230,7.502189,7.892376,10.438208,9.567538,8.763026,9.608265


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for dead or last follow up (days) or dead: 1
Number of samples with complete data for both variables: 95
Number of censored samples (event_var == 0.0): 74
Number of events (event_var == 1.0): 21
Censored ratio (among complete): 74 / 95 = 0.779 (77.9%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE181582/description.json
